# Unsteady DDES HLPW5

![HLPW5 geo](figures/HLPW5_geo.png)

Accurate predictions at large angles of attack (AOA) are crucial for aircraft CFD simulations, particularly for estimating stall speed and post-stall behavior. However, since RANS models rely on a time-averaged formulation, they inherently filter out flow unsteadiness and cannot capture phenomena such as vortex shedding, transient reattachment, and fluctuating separation bubbles. As a result, RANS often predicts flow separation either too early or too late, and the size and location of the recirculation region are typically inaccurate.

In this example, the HiLiftPW-5 (HLPW5) high-lift Common Research Model (CRM-HL) case is used to demonstrate the Flow360 workflow for performing an unsteady simulation using DDES to model high-AOA conditions where flow separation is expected. The volume mesh is provided directly in the Flow360 example library as the "High Lift Prediction Workshop 5" project, so it is loaded with `fl.Project.from_example` (shown below) and no manual mesh download or upload is required.

**Note:** The settings in this example are by no means a validation setup; they are crafted to showcase the capabilities of Flow360 and we have intentionally reduced node count and example FC cost. For rigorous validation, modify the settings as needed.

## 1. Create Project from the Example Library
In this example, the workflow begins with a volume mesh. The HLPW5 volume mesh is available directly in the Flow360 example library, so instead of uploading a local mesh file we copy the example project into the account with `fl.Project.from_example`. This creates a new project (with its own project ID) whose root asset is the volume mesh, which we then reference through `project.volume_mesh`. Because `from_example` copies the entire library project (including its reference case), we delete those copied cases first so the project starts clean.

In [1]:
import flow360 as fl

project = fl.Project.from_example(by_name="High Lift Prediction Workshop 5")

# from_example copies the whole library project, including its reference case.
# Delete those copied cases so the project starts clean and holds only the
# DDES case we submit below. The volume mesh (the project root) is untouched.
for case_id in project.get_case_ids():
    fl.Case.from_cloud(case_id).delete()

mesh_object = project.volume_mesh

[12:17:15] INFO: Copy operation started for project                             
           prj-5e29b718-4757-4d17-9f46-2648c428a44b. Waiting for completion...  


[12:17:31] INFO: Copy operation completed successfully.                         


## 2. Define time steps
In this step we define the unsteady time-stepping settings for the DDES simulation using `fl.Unsteady`. The code below specifies the number of physical steps, the physical time-step size, the maximum number of pseudo-steps per physical step, and the CFL strategy which in this case we use adaptive CFL.

In [2]:
time_stepping = fl.Unsteady(
    max_pseudo_steps=35,
    steps=600,
    step_size=0.001 * fl.u.s,
    CFL=fl.AdaptiveCFL(),  # Optionally switch to CFL=fl.RampCFL()
)

## 3. DDES setting 
The DDES (Delayed Detached Eddy Simulation) option enables a hybrid turbulence modeling approach suitable only for unsteady flow simulations. It is recommended for cases involving complex flow physics, such as significant separation regions or bluff body flows, as it provides higher solution fidelity than pure RANS models.
To use DDES, define the turbulence model as below:

In [3]:
turbulence_model_solver = fl.SpalartAllmaras(
    absolute_tolerance=1e-8,
    relative_tolerance=1e-2,
    linear_solver=fl.LinearSolver(max_iterations=25),
    hybrid_model=fl.DetachedEddySimulation(shielding_function="DDES"),
    rotation_correction=True,
    equation_evaluation_frequency=1,
)

## 4. Outputs
A dedicated output metric has been developed for DDES simulations, available as either `SpalartAllmaras_hybridModel` or `kOmegaSST_hybridModel`, depending on the turbulence model selected by the user. This metric provides five key DDES-related variables:

1) `f_d` – The shielding function that delineates the RANS and LES regions. When `f_d` = 0, the RANS model is fully applied; when `f_d` = 1, the LES model is used. Intermediate values represent a smooth transition between the two regimes.

2) `r_d` – A modified ratio of the modeled length scale to the wall distance, from which `f_d` is derived.

3) `DDES_lengthRANS` – The wall distance from the computational cell to the nearest solid boundary.

4) `DDES_lengthScale` -  The characteristic DES length scale $$\tilde{d} \equiv d - f_d \max(0, d - C_{DES}*\Delta)$$ 

5) `DDES_lengthLES` – The characteristic LES length scale, $$C_{DES}*\Delta$$

Among these variables, `f_d` is the most significant, as it enables users to identify and visualize the regions dominated by RANS and DES behavior within the computational domain.

![FD DDES](figures/fd_ddes.jpeg)

In [4]:
volume_outputs = fl.VolumeOutput(
    name="volume_outputs",
    output_fields=["SpalartAllmaras_hybridModel"],
    output_format=["tecplot"],
)

Additionally, for unsteady simulations, users may be interested in the time history of a flow field property. The following example shows a typical case where Q-criterion isosurfaces are output at a frequency of 20, with Mach number as the output field.

In [5]:
iso_surface = fl.IsosurfaceOutput(
    isosurfaces=[
        fl.Isosurface(
            name="Isosurface_Q_cri",
            iso_value=1e-6,
            field="qcriterion",
        ),
    ],
    output_format=["tecplot"],
    output_fields=["Mach"],
    frequency=20,
    frequency_offset=0,
)

## 5. Define Simulation Parameters
For Flow360 simulations, the operating conditions, boundary conditions (which depend on the surface names of the mesh), reference geometry, and solver settings can be defined in the same way as in other examples. For brevity, user can refer to those cases for detailed descriptions. The settings for this example are provided below, including the DDES configuration to illustrate the complete model setup.

In [6]:
with fl.SI_unit_system:
    params = fl.SimulationParams(
        time_stepping=time_stepping,
        operating_condition=fl.AerospaceCondition.from_mach(
            mach=0.2,
            alpha=17.05 * fl.u.deg,
            thermal_state=fl.ThermalState(
                temperature=289.44 * fl.u.K,
                density=0.002063 * fl.u.kg / fl.u.m**3,
                material=fl.Air(),
            ),
            reference_mach=0.2,
        ),
        reference_geometry=fl.ReferenceGeometry(),
        models=[
            fl.Wall(
                surfaces=[
                    mesh_object["fluid/FUSE"],
                    mesh_object["fluid/HORZ"],
                    mesh_object["fluid/CHINE"],
                    mesh_object["fluid/OBFLAP"],
                    mesh_object["fluid/IBSLAT"],
                    mesh_object["fluid/WING"],
                    mesh_object["fluid/VERT"],
                    mesh_object["fluid/SLAT_BKT"],
                    mesh_object["fluid/NAC"],
                    mesh_object["fluid/PYLON"],
                    mesh_object["fluid/IBFLAP"],
                    mesh_object["fluid/FSF"],
                    mesh_object["fluid/OBSLAT"],
                    mesh_object["fluid/WBF"],
                ],
            ),
            fl.Freestream(
                surfaces=[
                    mesh_object["fluid/Farfield"],
                    mesh_object["fluid/Inlet"],
                    mesh_object["fluid/Outlet"],
                ],
            ),
            fl.SlipWall(surfaces=mesh_object["fluid/Symmetry"]),
            fl.Fluid(
                navier_stokes_solver=fl.NavierStokesSolver(
                    absolute_tolerance=1e-10,
                    relative_tolerance=1e-2,
                    linear_solver=fl.LinearSolver(max_iterations=35),
                    kappa_MUSCL=-1,
                    numerical_dissipation_factor=1.0,
                ),
                turbulence_model_solver=turbulence_model_solver,
            ),
        ],
        outputs=[volume_outputs, iso_surface],
    )

[12:17:33] INFO: using: SI unit system for unit inference.                      


           NavierStokesSolver is deprecated; set them on the RoeFlux            
           riemann_solver instead.                                              


## 6. Run case
To run a case we need the previously defined `params` object to run the case in the project.

In [7]:
# `from_example` copies the example project at its original solver version
# (release-25.8). Pin the current release so the DDES case runs on a solver
# that matches the SimulationParams built above.
case = project.run_case(
    params=params,
    name="Unsteady DDES",
    solver_version="release-25.10",
)

[12:17:36] INFO: Successfully submitted:                                        
                   type        = Case                                           
                   name        = Unsteady DDES                                  
                   id          = case-8c621b40-4cc1-4c51-8ead-c980ee27b67a      
                   status      = pending                                        
                   project id  = prj-5e29b718-4757-4d17-9f46-2648c428a44b       
                                                                                


![qCriterion HLPW](figures/animation_iso_hlpw.gif)